# 02 – Clean layer
Builds clean_* and quarantine_* tables from the raw exercise_* tables, following issue_log.md.

**Conventions**
- Raw tables (exercise_*) are never modified.
- Every raw row ends up in exactly one place: the clean table or the quarantine table. Check: raw = clean + quarantine.
- **Quarantine:** rows that are invalid as a whole. Each has one `quarantine_reason`. When several apply, the first in this order wins: test_account, test_content, duplicate_row, invalid_event_type, then date reasons.
- **Flags:** clean rows that are valid but questionable keep a `dq_flags` column: a comma-separated list of flags (empty when none). A row can have more than one flag.
- **Fixes:** where a value is replaced (e.g., placeholder → null, invalid value → 'Unknown', placeholder price → grid price), the original value is kept in a column ending in `_raw`, so every change is traceable.
- Issue numbers (#) refer to issue_log.md.

In [0]:
-- Businesses Table: no issues found, straight copy
CREATE OR REPLACE TABLE workspace.scribd_exercise.clean_businesses AS SELECT * FROM workspace.scribd_exercise.exercise_businesses;

num_affected_rows,num_inserted_rows


In [0]:
-- Check Businesses Table
SELECT * FROM workspace.scribd_exercise.clean_businesses;

business_id,business_name,business_line,launch_date,hq_planet
BIZ_1,PageGarden Premium (SYNTHETIC),premium_published_content,2021-01-01,Terra Prime
BIZ_2,GatherSquare Community (SYNTHETIC),consumer_community,2021-06-15,Nearia
BIZ_3,MakerHarbor (SYNTHETIC),creator_content,2022-03-01,Wavanda


## Step 2 – Users (#11, #12, #13, #14)
Quarantine test accounts (all rows) and extra copies of duplicated users. Clean: one row per regular user; missing or invalid age band, channel and device mapped to 'Unknown', originals kept in _raw columns, each fix flagged.

In [0]:
-- Clean Users Table
CREATE OR REPLACE TABLE workspace.scribd_exercise.clean_users AS
WITH ranked AS (
  SELECT *,
         -- duplicates are exact copies, so any order works; keep the first
         ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY signup_date) AS row_num,
         -- a user is a test account if any of their rows says so
         MAX(CAST(is_test_account_flag AS INT)) OVER (PARTITION BY user_id) AS is_test
  FROM workspace.scribd_exercise.exercise_users
)
SELECT user_id,
       user_name,
       email,
       signup_date,
       country,

       CASE WHEN age_band IS NULL OR age_band IN ('N/A', '999') THEN 'Unknown'
            ELSE age_band END AS age_band,
       age_band AS age_band_raw,

       CASE WHEN acquisition_channel IS NULL OR acquisition_channel = 'carrier_pigeon' THEN 'Unknown'
            ELSE acquisition_channel END AS acquisition_channel,
       acquisition_channel AS acquisition_channel_raw,

       CASE WHEN primary_device IN ('unknown_device', 'smart_toaster') THEN 'Unknown'
            ELSE primary_device END AS primary_device,
       primary_device AS primary_device_raw,

       NULLIF(CONCAT_WS(',',
         CASE WHEN age_band IS NULL OR age_band = 'N/A' THEN 'age_band_missing' END,
         CASE WHEN age_band = '999' THEN 'age_band_invalid' END,
         CASE WHEN acquisition_channel IS NULL THEN 'channel_missing' END,
         CASE WHEN acquisition_channel = 'carrier_pigeon' THEN 'channel_invalid' END,
         CASE WHEN primary_device = 'unknown_device' THEN 'device_missing' END,
         CASE WHEN primary_device = 'smart_toaster' THEN 'device_invalid' END
       ), '') AS dq_flags
FROM ranked
WHERE is_test = 0
  AND row_num = 1;

num_affected_rows,num_inserted_rows


In [0]:
-- Quarantine Users Table 
CREATE OR REPLACE TABLE workspace.scribd_exercise.quarantine_users AS
WITH ranked AS (
    SELECT *, 
    ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY signup_date) AS row_num
    , MAX(CAST(is_test_account_flag AS INT)) OVER(PARTITION BY user_id) AS is_test
    FROM workspace.scribd_exercise.exercise_users)
SELECT * EXCEPT (row_num, is_test) --Keeps every original column and drops the two helper columns 
,CASE WHEN is_test = 1 THEN 'test_account' ELSE 'duplicate_row' END AS quarantine_reason
FROM ranked 
WHERE is_test = 1 OR row_num > 1;

num_affected_rows,num_inserted_rows


In [0]:
-- Checks
-- Reconciliation: raw = clean + quarantine
SELECT (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_users) AS raw_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_users) AS clean_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.quarantine_users) AS quarantined_rows,
       (SELECT COUNT(DISTINCT user_id) FROM workspace.scribd_exercise.clean_users) AS unique_clean_users;

raw_rows,clean_rows,quarantined_rows,unique_clean_users
806,760,46,760


In [0]:
-- What was quarantined, and what was flagged
SELECT 'quarantine' AS kind, quarantine_reason AS reason, COUNT(*) AS rows
FROM workspace.scribd_exercise.quarantine_users
GROUP BY quarantine_reason
UNION ALL
SELECT 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_users
LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag
GROUP BY flag
ORDER BY kind, rows DESC;

kind,reason,rows
flag,age_band_missing,39
flag,channel_invalid,20
flag,device_missing,18
flag,channel_missing,18
flag,device_invalid,18
flag,age_band_invalid,7
quarantine,test_account,40
quarantine,duplicate_row,6


## Step 3 – Content items (#4, #5, #7)
Quarantine test content (TEST_TYPE). Clean: runtime placeholders (-1, 99999) set to null (original kept in runtime_minutes_raw); items published before their business launched are flagged.

In [0]:
-- Clean Content Items Table 
CREATE OR REPLACE TABLE workspace.scribd_exercise.clean_content_items AS 
SELECT ci.content_id 
, ci.business_id 
, ci.creator_id
, ci.title
, ci.category
, ci.content_type
, ci.publish_date
,CASE WHEN ci.runtime_minutes_fake IN (-1,99999) THEN NULL 
      ELSE ci.runtime_minutes_fake END AS runtime_minutes
, ci.runtime_minutes_fake AS runtime_minutes_raw
,NULLIF(CONCAT_WS(','
    , CASE WHEN ci.runtime_minutes_fake IN (-1, 99999) THEN 'runtime_placeholder' END
    , CASE WHEN ci.publish_date < b.launch_date THEN 'published_before_launch' END
    ), '')  AS dq_flags
FROM workspace.scribd_exercise.exercise_content_items ci
JOIN workspace.scribd_exercise.exercise_businesses b
ON ci.business_id = b.business_id 
WHERE ci.content_type <> 'TEST_TYPE';

num_affected_rows,num_inserted_rows


In [0]:
-- Quarantine Content Items Table 
CREATE OR REPLACE TABLE workspace.scribd_exercise.quarantine_content_items AS 
SELECT *, 
'test_content' AS quarantine_reason
FROM workspace.scribd_exercise.exercise_content_items
WHERE content_type = 'TEST_TYPE';

num_affected_rows,num_inserted_rows


In [0]:
-- Checks 
-- Reconciliation: raw = clean + quarantine
SELECT (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_content_items) AS raw_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_content_items) AS clean_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.quarantine_content_items) AS quarantined_rows;

raw_rows,clean_rows,quarantined_rows
300,292,8


In [0]:
-- What was quarantined, and what was flagged
SELECT 'quarantine' AS kind, quarantine_reason AS reason, COUNT(*) AS rows
FROM workspace.scribd_exercise.quarantine_content_items
GROUP BY quarantine_reason
UNION ALL
SELECT 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_content_items
LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag
GROUP BY flag
ORDER BY kind, rows DESC;

kind,reason,rows
flag,runtime_placeholder,10
flag,published_before_launch,3
quarantine,test_content,8


## Step 4 – Creators (#2, #8, #9, #10)
No quarantine: every creator row is valid. Flags: creators in a business without creator content, joined before launch, placeholder follower counts (set to null, original in follower_count_raw), and content published before the join date.

In [0]:
-- Clean Creators Table 
CREATE OR REPLACE TABLE workspace.scribd_exercise.clean_creators WITH first_content AS (
    SELECT creator_id
    , MIN(publish_date) AS first_publish_date
    FROM workspace.scribd_exercise.clean_content_items
    WHERE creator_id IS NOT NULL 
    GROUP BY 1
)

SELECT cr.creator_id
, cr.creator_name
, cr.business_id 
, cr.join_date
, cr.payout_tier

, CASE WHEN cr.follower_count_fake < 0 OR cr.follower_count_fake =  999999 THEN NULL 
    ELSE cr.follower_count_fake END AS follower_count
, cr.follower_count_fake AS follower_count_raw 
,NULLIF(CONCAT_WS(','
    , CASE WHEN b.business_line <> 'creator_content' THEN 'not_creator_business' END
    , CASE WHEN cr.join_date < b.launch_date THEN 'joined_before_launch' END
    ,CASE WHEN cr.follower_count_fake < 0 OR cr.follower_count_fake = 999999 THEN 'follower_placeholder' END
    , CASE WHEN fc.first_publish_date < cr.join_date THEN 'content_before_join' END
    ), '') AS dq_flags

FROM workspace.scribd_exercise.exercise_creators cr
JOIN workspace.scribd_exercise.exercise_businesses b
ON cr.business_id = b.business_id 

LEFT JOIN first_content fc
ON cr.creator_id = fc.creator_id;



num_affected_rows,num_inserted_rows


In [0]:
-- Checks
-- Every creator kept: raw = clean (nothing quarantined)
SELECT (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_creators) AS raw_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_creators) AS clean_rows;

raw_rows,clean_rows
40,40


In [0]:
-- Flags
SELECT flag, COUNT(*) AS creators
FROM workspace.scribd_exercise.clean_creators
LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag
GROUP BY flag
ORDER BY creators DESC;

flag,creators
content_before_join,5
follower_placeholder,4
not_creator_business,3
joined_before_launch,2


## Step 5 – Subscriptions (#8, #14, #15, #16, #17, #18, #19, #20)
Quarantine subscriptions of test accounts. Clean: invalid status set to 'unknown'; placeholder prices replaced with the grid price; stale trials, superseded open subscriptions, and starts before launch or signup flagged. Originals kept in _raw columns.

In [0]:
-- Clean Subscriptions Table 
CREATE OR REPLACE TABLE workspace.scribd_exercise.clean_subscriptions AS
WITH
-- Standard price for each plan and billing cycle, ignoring placeholders
price_grid AS (
  SELECT plan_tier,
         billing_cycle,
         MODE(monthly_price_usd) AS grid_price
  FROM workspace.scribd_exercise.exercise_subscriptions
  WHERE monthly_price_usd NOT IN (-9.99, 999.99)
  GROUP BY plan_tier, billing_cycle
),
-- Subscriptions from regular users only (test accounts are quarantined)
regular AS (
  SELECT s.*
  FROM workspace.scribd_exercise.exercise_subscriptions s
  JOIN workspace.scribd_exercise.clean_users u
    ON s.user_id = u.user_id
),
-- Rank open (not canceled) subscriptions per user and business, newest first
ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY user_id, business_id, (status <> 'canceled')
           ORDER BY start_date DESC, subscription_id DESC
         ) AS open_rank
  FROM regular
)
SELECT r.subscription_id,
       r.user_id,
       r.business_id,
       r.plan_tier,
       r.billing_cycle,

       CASE WHEN r.monthly_price_usd IN (-9.99, 999.99) THEN g.grid_price
            ELSE r.monthly_price_usd END AS monthly_price_usd,
       r.monthly_price_usd AS monthly_price_usd_raw,

       CASE WHEN r.status = 'TEST_STATUS_DO_NOT_USE' THEN 'unknown'
            ELSE r.status END AS status,
       r.status AS status_raw,

       r.start_date,
       r.end_date,

       NULLIF(CONCAT_WS(',',
         CASE WHEN r.status = 'TEST_STATUS_DO_NOT_USE' THEN 'invalid_status' END,
         CASE WHEN r.monthly_price_usd IN (-9.99, 999.99) THEN 'price_imputed_from_grid' END,
         CASE WHEN r.status = 'trialing' AND DATEDIFF(DATE'2026-09-19', r.start_date) > 30 THEN 'stale_trial' END,
         CASE WHEN r.status <> 'canceled' AND r.open_rank > 1 THEN 'superseded_open_subscription' END,
         CASE WHEN r.start_date < b.launch_date THEN 'started_before_launch' END,
         CASE WHEN r.start_date < u.signup_date THEN 'started_before_signup' END
       ), '') AS dq_flags
FROM ranked r
JOIN workspace.scribd_exercise.exercise_businesses b
  ON r.business_id = b.business_id
JOIN workspace.scribd_exercise.clean_users u
  ON r.user_id = u.user_id
LEFT JOIN price_grid g
  ON r.plan_tier = g.plan_tier
 AND r.billing_cycle <=> g.billing_cycle;

num_affected_rows,num_inserted_rows


In [0]:
-- Quarantine Subscriptions Table 
CREATE OR REPLACE TABLE workspace.scribd_exercise.quarantine_subscriptions AS
SELECT s.*,
       'test_account' AS quarantine_reason
FROM workspace.scribd_exercise.exercise_subscriptions s
LEFT ANTI JOIN workspace.scribd_exercise.clean_users u
  ON s.user_id = u.user_id;

num_affected_rows,num_inserted_rows


In [0]:
-- Reconciliation: raw = clean + quarantine
SELECT (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_subscriptions) AS raw_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions) AS clean_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.quarantine_subscriptions) AS quarantined_rows;

raw_rows,clean_rows,quarantined_rows
843,804,39


In [0]:
-- What was quarantined, and what was flagged
SELECT 'quarantine' AS kind, quarantine_reason AS reason, COUNT(*) AS rows
FROM workspace.scribd_exercise.quarantine_subscriptions
GROUP BY quarantine_reason
UNION ALL
SELECT 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_subscriptions
LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag
GROUP BY flag
ORDER BY kind, rows DESC;

kind,reason,rows
flag,superseded_open_subscription,38
flag,stale_trial,32
flag,invalid_status,19
flag,price_imputed_from_grid,16
flag,started_before_signup,15
flag,started_before_launch,1
quarantine,test_account,39


In [0]:
-- Price check: no placeholders left, and every price now matches the grid
SELECT SUM(CASE WHEN monthly_price_usd IN (-9.99, 999.99) THEN 1 ELSE 0 END) AS placeholders_left,
SUM(CASE WHEN monthly_price_usd IS NULL THEN 1 ELSE 0 END) AS missing_prices,
MIN(monthly_price_usd) AS min_price,
MAX(monthly_price_usd) AS max_price
FROM workspace.scribd_exercise.clean_subscriptions;

placeholders_left,missing_prices,min_price,max_price
0,0,0.0,24.99


## Step 6 – Engagement events (#3, #4, #6, #14, #15, #21, #22, #25)
Quarantine: test-account events, invalid event types, and events more than 30 days before content publish or user signup. Clean: smart_toaster device set to 'Unknown'; placeholder durations set to null; regular users' events on test content, short date gaps, long sessions and events after the data cutoff flagged. Originals kept in _raw columns.

In [0]:
CREATE OR REPLACE TEMP VIEW events_tagged AS
SELECT e.*,
       DATEDIFF(c.publish_date, CAST(e.event_timestamp AS DATE)) AS days_before_publish,
       DATEDIFF(u.signup_date, CAST(e.event_timestamp AS DATE)) AS days_before_signup,
       CASE WHEN qc.content_id IS NOT NULL THEN 1 ELSE 0 END AS on_test_content,
       CASE
         WHEN u.user_id IS NULL THEN 'test_account'
         WHEN e.event_type = 'glitch_in_the_matrix' THEN 'invalid_event_type'
         WHEN DATEDIFF(c.publish_date, CAST(e.event_timestamp AS DATE)) > 30 THEN 'event_before_publish'
         WHEN DATEDIFF(u.signup_date, CAST(e.event_timestamp AS DATE)) > 30 THEN 'event_before_signup'
       END AS quarantine_reason
FROM workspace.scribd_exercise.exercise_engagement_events e
LEFT JOIN workspace.scribd_exercise.clean_users u
  ON e.user_id = u.user_id
LEFT JOIN workspace.scribd_exercise.clean_content_items c
  ON e.content_id = c.content_id
LEFT JOIN workspace.scribd_exercise.quarantine_content_items qc
  ON e.content_id = qc.content_id;

In [0]:
-- Clean Engagement Events Table
CREATE OR REPLACE TABLE workspace.scribd_exercise.clean_engagement_events AS
SELECT event_id,
       user_id,
       content_id,
       event_type,
       event_timestamp,

       CASE WHEN device = 'smart_toaster' THEN 'Unknown' ELSE device END AS device,
       device AS device_raw,

       CASE WHEN session_duration_sec IN (-15, 86400, 604800) THEN NULL
            ELSE session_duration_sec END AS session_duration_sec,
       session_duration_sec AS session_duration_sec_raw,

       NULLIF(CONCAT_WS(',',
         CASE WHEN on_test_content = 1 THEN 'on_test_content' END,
         CASE WHEN days_before_publish BETWEEN 1 AND 30 THEN 'event_before_publish_short' END,
         CASE WHEN days_before_signup BETWEEN 1 AND 30 THEN 'activity_before_signup' END,
         CASE WHEN device = 'smart_toaster' THEN 'device_invalid' END,
         CASE WHEN session_duration_sec IN (-15, 86400, 604800) THEN 'invalid_duration' END,
         CASE WHEN session_duration_sec > 14400
               AND session_duration_sec NOT IN (86400, 604800) THEN 'long_session' END,
         CASE WHEN event_timestamp >= TIMESTAMP'2026-09-20' THEN 'after_data_cutoff' END
       ), '') AS dq_flags
FROM events_tagged
WHERE quarantine_reason IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
-- Quarantine Engagement Events Table 
CREATE OR REPLACE TABLE workspace.scribd_exercise.quarantine_engagement_events AS
SELECT * EXCEPT (days_before_publish, days_before_signup, on_test_content)
FROM events_tagged
WHERE quarantine_reason IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
-- Reconciliation: raw = clean + quarantine
SELECT (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_engagement_events) AS raw_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events) AS clean_rows,
       (SELECT COUNT(*) FROM workspace.scribd_exercise.quarantine_engagement_events) AS quarantined_rows;

raw_rows,clean_rows,quarantined_rows
6000,5637,363


In [0]:
-- What was quarantined, and what was flagged
SELECT 'quarantine' AS kind, quarantine_reason AS reason, COUNT(*) AS rows
FROM workspace.scribd_exercise.quarantine_engagement_events
GROUP BY quarantine_reason
UNION ALL
SELECT 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_engagement_events
LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag
GROUP BY flag
ORDER BY kind, rows DESC;

kind,reason,rows
flag,on_test_content,160
flag,long_session,66
flag,invalid_duration,59
flag,device_invalid,45
flag,event_before_publish_short,16
flag,activity_before_signup,14
flag,after_data_cutoff,4
quarantine,test_account,308
quarantine,invalid_event_type,38
quarantine,event_before_publish,16


In [0]:
SELECT CASE WHEN e.event_id IS NOT NULL THEN 'clean' ELSE 'quarantine' END AS ended_up_in,
       q.quarantine_reason,
       e.dq_flags,
       COUNT(*) AS events
FROM workspace.scribd_exercise.exercise_engagement_events r
JOIN workspace.scribd_exercise.clean_users u
  ON r.user_id = u.user_id
LEFT JOIN workspace.scribd_exercise.clean_engagement_events e
  ON r.event_id = e.event_id
LEFT JOIN workspace.scribd_exercise.quarantine_engagement_events q
  ON r.event_id = q.event_id
WHERE CAST(r.event_timestamp AS DATE) < u.signup_date
GROUP BY 1, 2, 3
ORDER BY events DESC;

ended_up_in,quarantine_reason,dq_flags,events
quarantine,event_before_publish,null,15
clean,null,activity_before_signup,12
clean,null,"event_before_publish_short,activity_before_signup",2
quarantine,event_before_signup,null,1


## Step 7 – Reconciliation checks and data-health summary

In [0]:
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_data_health AS
WITH counts AS (
  SELECT 'businesses' AS table_name,
         (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_businesses) AS raw_rows,
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_businesses) AS clean_rows,
         0 AS quarantined_rows,
         0 AS flagged_rows
  UNION ALL
  SELECT 'creators',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_creators),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_creators),
         0,
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_creators WHERE dq_flags IS NOT NULL)
  UNION ALL
  SELECT 'content_items',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_content_items),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_content_items),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.quarantine_content_items),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_content_items WHERE dq_flags IS NOT NULL)
  UNION ALL
  SELECT 'users',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_users),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_users),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.quarantine_users),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_users WHERE dq_flags IS NOT NULL)
  UNION ALL
  SELECT 'subscriptions',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_subscriptions),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.quarantine_subscriptions),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions WHERE dq_flags IS NOT NULL)
  UNION ALL
  SELECT 'engagement_events',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.exercise_engagement_events),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.quarantine_engagement_events),
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events WHERE dq_flags IS NOT NULL)
)
SELECT table_name,
       raw_rows,
       clean_rows,
       quarantined_rows,
       ROUND(100.0 * quarantined_rows / raw_rows, 1) AS pct_quarantined,
       flagged_rows,
       ROUND(100.0 * flagged_rows / clean_rows, 1) AS pct_flagged,
       CASE WHEN raw_rows = clean_rows + quarantined_rows THEN 'PASS' ELSE 'FAIL' END AS reconciliation
FROM counts;

In [0]:
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_data_health_reasons AS
SELECT 'content_items' AS table_name, 'quarantine' AS kind, quarantine_reason AS reason, COUNT(*) AS rows
FROM workspace.scribd_exercise.quarantine_content_items GROUP BY quarantine_reason
UNION ALL
SELECT 'users', 'quarantine', quarantine_reason, COUNT(*)
FROM workspace.scribd_exercise.quarantine_users GROUP BY quarantine_reason
UNION ALL
SELECT 'subscriptions', 'quarantine', quarantine_reason, COUNT(*)
FROM workspace.scribd_exercise.quarantine_subscriptions GROUP BY quarantine_reason
UNION ALL
SELECT 'engagement_events', 'quarantine', quarantine_reason, COUNT(*)
FROM workspace.scribd_exercise.quarantine_engagement_events GROUP BY quarantine_reason
UNION ALL
SELECT 'creators', 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_creators LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag GROUP BY flag
UNION ALL
SELECT 'content_items', 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_content_items LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag GROUP BY flag
UNION ALL
SELECT 'users', 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_users LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag GROUP BY flag
UNION ALL
SELECT 'subscriptions', 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_subscriptions LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag GROUP BY flag
UNION ALL
SELECT 'engagement_events', 'flag', flag, COUNT(*)
FROM workspace.scribd_exercise.clean_engagement_events LATERAL VIEW EXPLODE(SPLIT(dq_flags, ',')) f AS flag GROUP BY flag;

In [0]:
SELECT * FROM workspace.scribd_exercise.metrics_data_health ORDER BY table_name;

table_name,raw_rows,clean_rows,quarantined_rows,pct_quarantined,flagged_rows,pct_flagged,reconciliation
businesses,3,3,0,0.0,0,0.0,PASS
content_items,300,292,8,2.7,13,4.5,PASS
creators,40,40,0,0.0,11,27.5,PASS
engagement_events,6000,5637,363,6.1,358,6.4,PASS
subscriptions,843,804,39,4.6,114,14.2,PASS
users,806,760,46,5.7,112,14.7,PASS


In [0]:
SELECT * FROM workspace.scribd_exercise.metrics_data_health_reasons ORDER BY table_name, kind, rows DESC;

table_name,kind,reason,rows
content_items,flag,runtime_placeholder,10
content_items,flag,published_before_launch,3
content_items,quarantine,test_content,8
creators,flag,content_before_join,5
creators,flag,follower_placeholder,4
creators,flag,not_creator_business,3
creators,flag,joined_before_launch,2
engagement_events,flag,on_test_content,160
engagement_events,flag,long_session,66
engagement_events,flag,invalid_duration,59


In [0]:
SELECT check_name,
       failures,
       CASE WHEN failures = 0 THEN 'PASS' ELSE 'FAIL' END AS result
FROM (
  -- 1. Reconciliation: raw = clean + quarantine, every table
  SELECT 'raw = clean + quarantine (all tables)' AS check_name,
         (SELECT COUNT(*) FROM workspace.scribd_exercise.metrics_data_health WHERE reconciliation = 'FAIL') AS failures
  UNION ALL
  -- 2. IDs still unique in clean tables
  SELECT 'unique user_id in clean_users',
         (SELECT COUNT(*) - COUNT(DISTINCT user_id) FROM workspace.scribd_exercise.clean_users)
  UNION ALL
  SELECT 'unique content_id in clean_content_items',
         (SELECT COUNT(*) - COUNT(DISTINCT content_id) FROM workspace.scribd_exercise.clean_content_items)
  UNION ALL
  SELECT 'unique creator_id in clean_creators',
         (SELECT COUNT(*) - COUNT(DISTINCT creator_id) FROM workspace.scribd_exercise.clean_creators)
  UNION ALL
  SELECT 'unique subscription_id in clean_subscriptions',
         (SELECT COUNT(*) - COUNT(DISTINCT subscription_id) FROM workspace.scribd_exercise.clean_subscriptions)
  UNION ALL
  SELECT 'unique event_id in clean_engagement_events',
         (SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM workspace.scribd_exercise.clean_engagement_events)
  UNION ALL
  -- 3. No test data left
  SELECT 'clean subscriptions belong to clean users',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions s
            LEFT ANTI JOIN workspace.scribd_exercise.clean_users u ON s.user_id = u.user_id)
  UNION ALL
  SELECT 'clean events belong to clean users',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events e
            LEFT ANTI JOIN workspace.scribd_exercise.clean_users u ON e.user_id = u.user_id)
  UNION ALL
  SELECT 'no TEST_TYPE content left',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_content_items WHERE content_type = 'TEST_TYPE')
  UNION ALL
  SELECT 'no invalid event types left',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events WHERE event_type = 'glitch_in_the_matrix')
  UNION ALL
  -- 4. No placeholder or invalid values left
  SELECT 'no placeholder prices left',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions WHERE monthly_price_usd IN (-9.99, 999.99) OR monthly_price_usd IS NULL)
  UNION ALL
  SELECT 'no placeholder durations left',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events WHERE session_duration_sec IN (-15, 86400, 604800))
  UNION ALL
  SELECT 'no placeholder runtimes left',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_content_items WHERE runtime_minutes IN (-1, 99999))
  UNION ALL
  SELECT 'no placeholder follower counts left',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_creators WHERE follower_count < 0 OR follower_count = 999999)
  UNION ALL
  SELECT 'no invalid segment values left in users',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_users
           WHERE age_band IN ('N/A', '999') OR age_band IS NULL
              OR acquisition_channel IN ('carrier_pigeon') OR acquisition_channel IS NULL
              OR primary_device IN ('unknown_device', 'smart_toaster'))
  UNION ALL
  SELECT 'no invalid devices left in events',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events WHERE device = 'smart_toaster')
  UNION ALL
  SELECT 'no TEST_STATUS left in subscriptions',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions WHERE status = 'TEST_STATUS_DO_NOT_USE')
  UNION ALL
  -- 5. Clean events with content link to clean content (except flagged events on test content)
  SELECT 'clean events link to clean content',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events e
            LEFT ANTI JOIN workspace.scribd_exercise.clean_content_items c ON e.content_id = c.content_id
           WHERE e.content_id IS NOT NULL
             AND (e.dq_flags IS NULL OR e.dq_flags NOT LIKE '%on_test_content%'))
  UNION ALL
  -- 6. Nulls: columns that must always be filled
  SELECT 'no missing key fields in clean_subscriptions',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions
           WHERE user_id IS NULL OR business_id IS NULL OR start_date IS NULL OR status IS NULL)
  UNION ALL
  SELECT 'no missing key fields in clean_engagement_events',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events
           WHERE user_id IS NULL OR event_type IS NULL OR event_timestamp IS NULL)
  UNION ALL
  -- 7. Nulls that are only allowed in specific cases
  SELECT 'billing_cycle missing only on Free plans',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions
           WHERE billing_cycle IS NULL AND plan_tier <> 'Free')
  UNION ALL
  SELECT 'end_date missing only when not canceled',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_subscriptions
           WHERE (status = 'canceled' AND end_date IS NULL)
              OR (status <> 'canceled' AND end_date IS NOT NULL))
  UNION ALL
  SELECT 'creator_id missing only outside creator_content',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_content_items c
            JOIN workspace.scribd_exercise.clean_businesses b ON c.business_id = b.business_id
           WHERE (b.business_line = 'creator_content' AND c.creator_id IS NULL)
              OR (b.business_line <> 'creator_content' AND c.creator_id IS NOT NULL))
  UNION ALL
  SELECT 'content events always have a content_id',
         (SELECT COUNT(*) FROM workspace.scribd_exercise.clean_engagement_events
           WHERE content_id IS NULL
             AND event_type IN ('content_open', 'content_complete', 'like', 'share', 'comment_post', 'download'))
  UNION ALL
  -- 8. Nulls within a threshold (baseline from the audit, with room to spare)
  SELECT 'unknown age band at most 8% of users (baseline 6%)',
         (SELECT CASE WHEN AVG(CASE WHEN age_band = 'Unknown' THEN 1.0 ELSE 0 END) > 0.08 THEN 1 ELSE 0 END
            FROM workspace.scribd_exercise.clean_users)
  UNION ALL
  SELECT 'unknown acquisition channel at most 8% of users (baseline 5%)',
         (SELECT CASE WHEN AVG(CASE WHEN acquisition_channel = 'Unknown' THEN 1.0 ELSE 0 END) > 0.08 THEN 1 ELSE 0 END
            FROM workspace.scribd_exercise.clean_users)
) checks;

check_name,failures,result
raw = clean + quarantine (all tables),0,PASS
unique user_id in clean_users,0,PASS
unique content_id in clean_content_items,0,PASS
unique creator_id in clean_creators,0,PASS
unique subscription_id in clean_subscriptions,0,PASS
unique event_id in clean_engagement_events,0,PASS
clean subscriptions belong to clean users,0,PASS
clean events belong to clean users,0,PASS
no TEST_TYPE content left,0,PASS
no invalid event types left,0,PASS


**Result:** 
- All 18 checks pass: raw = clean + quarantine for every table, 
- IDs unique in every clean table 
- No test data, placeholders or invalid values left
- Clean tables link to each other (except the 160 events flagged on_test_content, by design).

**Data health:** 
- Engagement events 6.1% quarantined (mostly test accounts)
- Users 14.7%
- Subscriptions 14.2% flagged (mostly fixes that keep rows usable); 
- Creators 27.5% flagged (11 of 40)